# 01 -- Data Cleaning & Merging

Cilem tohoto notebooku je:
1. Nacist a vycistit cenova data akcii (yfinance)
2. Nacist a vycistit Reddit sentiment data (historicky Kaggle dataset)
3. Otagovat Reddit posty podle zminovanych tickeru
4. Agregovat pocet zminek na denni urovni per ticker
5. Ulozit vycistena data do `data/processed/` pro dalsi analyzu

**Dulezita poznamka k metodologii:** V tomto notebooku zatim neresime spojeni
cen a sentimentu do jedne casove osy -- to delame az v dalsim notebooku,
kde budeme muset dat extra pozor na *look-ahead bias* (sentiment k danemu
dni musi byt dostupny pred cenovym pohybem, ktery se snazime vysvetlit,
ne po nem).

## Import knihoven

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Cesty k datum

Notebook lezi v `notebooks/`, takze vsechny cesty jdou o uroven vys.

In [2]:
PROJECT_ROOT = Path.cwd().parent
PRICES_DIR = PROJECT_ROOT / "data" / "raw" / "prices"
REDDIT_DIR = PROJECT_ROOT / "data" / "raw" / "reddit"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Prices dir:  {PRICES_DIR}")
print(f"Reddit dir:  {REDDIT_DIR}")
print(f"Processed:   {PROCESSED_DIR}")

Prices dir:  /Users/maxkubicek/Desktop/market-pulse/data/raw/prices
Reddit dir:  /Users/maxkubicek/Desktop/market-pulse/data/raw/reddit
Processed:   /Users/maxkubicek/Desktop/market-pulse/data/processed


## 1. Nacteni cenovych dat

Kazdy ticker ma svuj CSV v `data/raw/prices/`. Nacteme vsechny a spojime
do jednoho DataFrame se sloupcem `Ticker`.

In [3]:
price_files = sorted(PRICES_DIR.glob("*.csv"))
print(f"Nalezeno {len(price_files)} souboru:")
for f in price_files:
    print(f"  {f.name}")

Nalezeno 9 souboru:
  AAPL.csv
  AMC.csv
  AMZN.csv
  DIS.csv
  GME.csv
  MSFT.csv
  NVDA.csv
  PLTR.csv
  TSLA.csv


In [4]:
price_dfs = []

for f in price_files:
    df = pd.read_csv(f)
    price_dfs.append(df)

prices = pd.concat(price_dfs, ignore_index=True)
print(f"Celkem radku: {len(prices)}")
prices.head()

Celkem radku: 14644


,Date,Close,High,Low,Open,Volume,Ticker
0,2020-01-02,72.333893,72.394101,71.091199,71.344069,135480400,AAPL
1,2020-01-03,71.630638,72.389257,71.406666,71.563205,146322800,AAPL
2,2020-01-06,72.201416,72.239950,70.503554,70.754021,118387200,AAPL
3,2020-01-07,71.861832,72.466315,71.642674,72.211033,108872000,AAPL
4,2020-01-08,73.017830,73.318870,71.565614,71.565614,132079200,AAPL


### Cisteni cenovych dat

Par veci, ktere je potreba osetrit:
- sloupec `Date` je zatim text, prevedeme na datetime
- zkontrolujeme chybejici hodnoty (yfinance obcas vraci NaN u objemu/ceny
  pro dny, kdy se s akcii neobchodovalo, napr. svatky)
- serad'ime podle tickeru a data

In [5]:
prices["Date"] = pd.to_datetime(prices["Date"])

print("Chybejici hodnoty po sloupcich:")
print(prices.isna().sum())

Chybejici hodnoty po sloupcich:
Date      0
Close     9
High      9
Low       9
Open      9
Volume    0
Ticker    0
dtype: int64


In [6]:
# Pokud existuji radky s chybejicimi cenami, podivame se na ne blize
# nez se rozhodneme, jak je osetrit (smazat vs. dopocitat)
missing_rows = prices[prices.isna().any(axis=1)]
missing_rows

,Date,Close,High,Low,Open,Volume,Ticker
1647,2026-07-24,NaN,NaN,NaN,NaN,47460975,AAPL
3295,2026-07-24,NaN,NaN,NaN,NaN,45222927,AMC
4943,2026-07-24,NaN,NaN,NaN,NaN,34991618,AMZN
6591,2026-07-24,NaN,NaN,NaN,NaN,9944679,DIS
8239,2026-07-24,NaN,NaN,NaN,NaN,2204907,GME
9887,2026-07-24,NaN,NaN,NaN,NaN,27640155,MSFT
11535,2026-07-24,NaN,NaN,NaN,NaN,114728583,NVDA
12995,2026-07-24,NaN,NaN,NaN,NaN,21150618,PLTR
14643,2026-07-24,NaN,NaN,NaN,NaN,62648724,TSLA


In [7]:
# Pokud jsou chybejici radky jen ojedinele (napr. svatky), je bezpecne
# je odstranit -- pro analyzu sentiment vs. cena nam chybejici den nevadi,
# proste pro nej nebudeme mit signal
prices = prices.dropna()

prices = prices.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Po cisteni: {len(prices)} radku")
prices.head()

Po cisteni: 14635 radku


,Date,Close,High,Low,Open,Volume,Ticker
0,2020-01-02,72.333893,72.394101,71.091199,71.344069,135480400,AAPL
1,2020-01-03,71.630638,72.389257,71.406666,71.563205,146322800,AAPL
2,2020-01-06,72.201416,72.239950,70.503554,70.754021,118387200,AAPL
3,2020-01-07,71.861832,72.466315,71.642674,72.211033,108872000,AAPL
4,2020-01-08,73.017830,73.318870,71.565614,71.565614,132079200,AAPL


In [8]:
# Rychla kontrola casoveho rozpeti per ticker -- pripomenme si, ze PLTR
# ma kratsi historii kvuli pozdejsimu IPO (zari 2020)
prices.groupby("Ticker")["Date"].agg(["min", "max", "count"])

,min,max,count
Ticker,,,
AAPL,2020-01-02,2026-07-23,1647
AMC,2020-01-02,2026-07-23,1647
AMZN,2020-01-02,2026-07-23,1647
DIS,2020-01-02,2026-07-23,1647
GME,2020-01-02,2026-07-23,1647
MSFT,2020-01-02,2026-07-23,1647
NVDA,2020-01-02,2026-07-23,1647
PLTR,2020-09-30,2026-07-23,1459
TSLA,2020-01-02,2026-07-23,1647


### Vypocet dennich vynosu a volatility

Pro pozdejsi backtest budeme potrebovat denni procentualni zmenu ceny
(return) a klouzavou volatilitu -- pridame je uz ted, at je mame hotove.

In [9]:
prices["Daily_Return"] = prices.groupby("Ticker")["Close"].pct_change()

# 5-denni klouzava volatilita (smerodatna odchylka vynosu) jako jednoducha
# proxy pro "jak moc se dana akcie prave hejbe"
prices["Volatility_5d"] = (
    prices.groupby("Ticker")["Daily_Return"]
    .transform(lambda x: x.rolling(window=5).std())
)

prices.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return,Volatility_5d
0,2020-01-02,72.333893,72.394101,71.091199,71.344069,135480400,AAPL,NaN,NaN
1,2020-01-03,71.630638,72.389257,71.406666,71.563205,146322800,AAPL,-0.009722,NaN
2,2020-01-06,72.201416,72.239950,70.503554,70.754021,118387200,AAPL,0.007968,NaN
3,2020-01-07,71.861832,72.466315,71.642674,72.211033,108872000,AAPL,-0.004703,NaN
4,2020-01-08,73.017830,73.318870,71.565614,71.565614,132079200,AAPL,0.016086,NaN
5,2020-01-09,74.568771,74.761434,73.742720,73.993180,170108400,AAPL,0.021241,0.013224
6,2020-01-10,74.737366,75.300919,74.236439,74.802395,140644800,AAPL,0.002261,0.010409
7,2020-01-13,76.334091,76.360587,74.934858,75.052871,121532000,AAPL,0.021364,0.011841
8,2020-01-14,75.303329,76.481001,75.180510,76.271479,161954400,AAPL,-0.013503,0.015029
9,2020-01-15,74.980621,75.982483,74.549530,75.103448,121923600,AAPL,-0.004285,0.015547


## 2. Nacteni Reddit dat

Nacteme historicky dataset a rovnou opravime dva problemy, na ktere jsme
narazili pri inspekci:
1. `created_utc` je Unix timestamp v sekundach (ne nanosekundach)
2. matching tickeru musi byt na cela slova, ne podretezce (jinak napr.
   "KO" chytne cast slova "looking")

In [10]:
reddit_files = list(REDDIT_DIR.glob("*.csv"))
assert len(reddit_files) >= 1, "Nenalezen zadny CSV v data/raw/reddit/"
reddit_path = reddit_files[0]
print(f"Nacitam: {reddit_path.name}")

reddit = pd.read_csv(reddit_path, low_memory=False)
print(f"Celkem radku: {len(reddit)}")
reddit.head()

Nacitam: r_wallstreetbets_posts.csv
Celkem radku: 1118863


,id,title,score,author,author_flair_text,removed_by,total_awards_received,awarders,created_utc,full_link,num_comments,over_18
0,ll0n5k,Whats going on with PLTR?,1,Zaccko98,NaN,moderator,0.0,[],1613469192,https://www.reddit.com/r/wallstreetbets/commen...,2,False
1,ll0n4p,"Need explanations on Level 2 data for GME, why...",1,toutoucnc,210115:1:1,moderator,0.0,[],1613469189,https://www.reddit.com/r/wallstreetbets/commen...,2,False
2,ll0my2,XRT is being used as a laundry short machine,1,thabat,NaN,moderator,0.0,[],1613469166,https://www.reddit.com/r/wallstreetbets/commen...,2,False
3,ll0mxr,Airlines?,1,AsianTwink_,NaN,moderator,0.0,[],1613469165,https://www.reddit.com/r/wallstreetbets/commen...,2,False
4,ll0mx4,Buy TRXC 🚀,1,Oneverystreet8,NaN,moderator,0.0,[],1613469164,https://www.reddit.com/r/wallstreetbets/commen...,2,False


In [11]:
reddit["created_utc"] = pd.to_datetime(reddit["created_utc"], unit="s", utc=True)

# Pro spojeni s cenovymi daty (ktera jsou po dnech, bez casove zony)
# potrebujeme jen datum, ne presny cas
reddit["Date"] = reddit["created_utc"].dt.tz_localize(None).dt.normalize()

print(f"Casove rozpeti: {reddit['Date'].min()} az {reddit['Date'].max()}")

Casove rozpeti: 2012-04-11 00:00:00 az 2021-02-16 00:00:00


### Oriznuti na relevantni obdobi

Cenova data mame od 2020-01-01, takze Reddit data z let 2012-2019 nam
k nicemu nebudou -- oriznuti hned na zacatku setri pamet i cas pri
dalsim zpracovani.

In [12]:
PRICE_START = prices["Date"].min()
print(f"Cenova data zacinaji: {PRICE_START}")

reddit = reddit[reddit["Date"] >= PRICE_START].reset_index(drop=True)
print(f"Reddit radku po oriznuti: {len(reddit)}")

Cenova data zacinaji: 2020-01-02 00:00:00
Reddit radku po oriznuti: 865352


### Tagovani postu podle tickeru

Pouzivame stejny pristup jako v inspekcnim skriptu -- hledame cele slovo
(alias nazvu firmy nebo symbol tickeru), ne podretezec. Jeden post muze
zminovat vic tickeru najednou, takze vysledek bude "dlouhy" format:
jeden radek = (post, ticker), ne (post).

In [13]:
TICKER_ALIASES = {
    "AAPL": ["AAPL", "Apple"],
    "TSLA": ["TSLA", "Tesla"],
    "NVDA": ["NVDA", "Nvidia"],
    "MSFT": ["MSFT", "Microsoft"],
    "AMZN": ["AMZN", "Amazon"],
    "GME":  ["GME", "Gamestop", "GameStop"],
    "AMC":  ["AMC"],
    "PLTR": ["PLTR", "Palantir"],
    "DIS":  ["DIS", "Disney"],
}

# Predkompilovane regexy pro rychlost -- vsechny aliasy pro dany ticker
# spojime do jednoho patternu pomoci OR (|)
ticker_patterns = {
    ticker: re.compile(
        r"\b(" + "|".join(re.escape(a.lower()) for a in aliases) + r")\b"
    )
    for ticker, aliases in TICKER_ALIASES.items()
}

In [14]:
reddit["title_lower"] = reddit["title"].astype(str).str.lower()

# Pro kazdy ticker vytvorime bool sloupec, jestli se v postu zminuje
for ticker, pattern in ticker_patterns.items():
    reddit[f"mentions_{ticker}"] = reddit["title_lower"].str.contains(pattern, na=False)

mention_cols = [f"mentions_{t}" for t in TICKER_ALIASES]
reddit["any_mention"] = reddit[mention_cols].any(axis=1)

print(f"Postu s alespon jednou relevantni zminkou: {reddit['any_mention'].sum()} z {len(reddit)}")

/var/folders/c3/7nw705bn2xv4mr97p535fpp80000gn/T/ipykernel_56951/4130701103.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  reddit[f"mentions_{ticker}"] = reddit["title_lower"].str.contains(pattern, na=False)
/var/folders/c3/7nw705bn2xv4mr97p535fpp80000gn/T/ipykernel_56951/4130701103.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  reddit[f"mentions_{ticker}"] = reddit["title_lower"].str.contains(pattern, na=False)
/var/folders/c3/7nw705bn2xv4mr97p535fpp80000gn/T/ipykernel_56951/4130701103.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  reddit[f"mentions_{ticker}"] = reddit["title_lower"].str.contains(pattern, na=False)
/var/folders/c3/7nw705bn2xv4mr97p535fpp80000gn/T/ipykernel_56951/4130701103.py:5: UserWarning:

Postu s alespon jednou relevantni zminkou: 164590 z 865352


/var/folders/c3/7nw705bn2xv4mr97p535fpp80000gn/T/ipykernel_56951/4130701103.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  reddit[f"mentions_{ticker}"] = reddit["title_lower"].str.contains(pattern, na=False)


In [15]:
# Prevod na "long" format: jeden radek = (post, ticker)
# tohle nam usnadni pozdejsi agregaci per ticker per den
relevant = reddit[reddit["any_mention"]].copy()

long_rows = []
for ticker in TICKER_ALIASES:
    subset = relevant[relevant[f"mentions_{ticker}"]][["id", "Date", "title", "score", "num_comments"]].copy()
    subset["Ticker"] = ticker
    long_rows.append(subset)

reddit_long = pd.concat(long_rows, ignore_index=True)
print(f"Celkem (post, ticker) paru: {len(reddit_long)}")
reddit_long.head()

Celkem (post, ticker) paru: 185814


,id,Date,title,score,num_comments,Ticker
0,lkzz8c,2021-02-16,What's the future ? Will they ever coop large ...,1,0,AAPL
1,lkw2er,2021-02-16,Apple car,1,1,AAPL
2,lkvwy5,2021-02-16,Apple iCar,1,1,AAPL
3,lkvr83,2021-02-16,"M1 chip is fukt, $AAPL Puts?",1,0,AAPL
4,lkor1v,2021-02-15,"A Huge Fund Bought Tesla, Apple, and Microsoft...",1,0,AAPL


### Agregace na denni urovni

Pro kazdy ticker a den spocitame:
- pocet postu (`post_count`) -- proxy pro "objem pozornosti"
- soucet skore a komentaru -- proxy pro "angazovanost" komunity

Skutecny sentiment (pozitivni/negativni) spocitame az v dalsim notebooku
pomoci FinBERT -- tady zatim jen pripravujeme strukturu dat.

In [16]:
daily_mentions = (
    reddit_long
    .groupby(["Ticker", "Date"])
    .agg(
        post_count=("id", "count"),
        total_score=("score", "sum"),
        total_comments=("num_comments", "sum"),
    )
    .reset_index()
    .sort_values(["Ticker", "Date"])
)

daily_mentions.head(10)

,Ticker,Date,post_count,total_score,total_comments
0,AAPL,2020-01-02,7,139,194
1,AAPL,2020-01-03,6,26,49
2,AAPL,2020-01-04,2,10651,338
3,AAPL,2020-01-05,1,1,0
4,AAPL,2020-01-06,2,2,24
5,AAPL,2020-01-08,4,4,36
6,AAPL,2020-01-09,16,16,185
7,AAPL,2020-01-10,6,6,118
8,AAPL,2020-01-11,4,4,268
9,AAPL,2020-01-12,4,4,32


In [17]:
# Rychla kontrola pokryti -- kolik dni ma kazdy ticker alespon 1 zminku
daily_mentions.groupby("Ticker")["Date"].count().sort_values(ascending=False)

Ticker
TSLA    407
AAPL    377
AMZN    360
MSFT    320
DIS     298
GME     269
NVDA    221
AMC     186
PLTR    153
Name: Date, dtype: int64

## 3. Ulozeni vycistenych dat

Ukladame dve samostatne tabulky do `data/processed/`:
- `prices_clean.csv` -- denni ceny + returns + volatilita
- `reddit_daily_mentions.csv` -- denni pocty zminek per ticker

Spojeni obou do jedne casove osy (a reseni look-ahead bias) budeme delat
az v dalsim notebooku, protoze je to samostatny, dulezity krok, ktery
si zaslouzi vlastni prostor a peclivou kontrolu.

In [18]:
prices.to_csv(PROCESSED_DIR / "prices_clean.csv", index=False)
daily_mentions.to_csv(PROCESSED_DIR / "reddit_daily_mentions.csv", index=False)

# Ukladame i post-level data s textem -- budeme je potrebovat v notebooku 02
# pro FinBERT sentiment scoring (tam uz nestaci jen agregovane pocty,
# potrebujeme skutecny text kazdeho postu)
reddit_long.to_csv(PROCESSED_DIR / "reddit_posts_tagged.csv", index=False)

print("Ulozeno:")
print(f"  {PROCESSED_DIR / 'prices_clean.csv'}  ({len(prices)} radku)")
print(f"  {PROCESSED_DIR / 'reddit_daily_mentions.csv'}  ({len(daily_mentions)} radku)")
print(f"  {PROCESSED_DIR / 'reddit_posts_tagged.csv'}  ({len(reddit_long)} radku)")

Ulozeno:
  /Users/maxkubicek/Desktop/market-pulse/data/processed/prices_clean.csv  (14635 radku)
  /Users/maxkubicek/Desktop/market-pulse/data/processed/reddit_daily_mentions.csv  (2591 radku)
  /Users/maxkubicek/Desktop/market-pulse/data/processed/reddit_posts_tagged.csv  (185814 radku)


## Shrnuti

- Cenova data: {n_tickers} tickeru, ocistena, s dopocitanymi returns a volatilitou
- Reddit data: orizuta na relevantni obdobi, otagovana podle tickeru, agregovana na denni urovni
- Obe tabulky ulozeny do `data/processed/` a pripravene na spojeni

**Dalsi krok (notebook 02):** spojeni obou datasetu na spolecnou casovou
osu s dukladnym osetrenim look-ahead bias, a prvni pohled na to, jestli
mezi vykyvy v poctu zminek a nasledujicimi cenovymi pohyby vubec neco je.